# Temporal Fusion Transformer (TFT) Optimization with Optuna
## Inventory Management & Supply Chain Forecasting

This notebook performs end-to-end hyperparameter optimization for TFT using **Optuna (TPE Bayesian Optimization)** with **Huber Loss** and **direct multi-step temporal decoding**.


In [ ]:
# Cell 1: Environment & GPU Verification
import torch
import sys

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("Warning: GPU not detected. Please select Runtime > Change runtime type > T4 GPU in Colab.")


In [ ]:
# Cell 2: Install Optuna and Required Packages
!pip install -q optuna joblib scikit-learn

import optuna
print(f"Optuna Version: {optuna.__version__}")


In [ ]:
# Cell 3: Unzip and Verify TFT Sequence Files
import zipfile
import os
from pathlib import Path

# Adjust zip_path if uploaded under a different filename
zip_path = "/content/tft.zip"
extract_path = Path("/content/tft")

if os.path.exists(zip_path):
    print(f"Extracting {zip_path} to {extract_path}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
elif extract_path.exists():
    print(f"Directory {extract_path} already exists. Proceeding.")
else:
    raise FileNotFoundError(
        f"Could not find {zip_path} or directory {extract_path}. "
        "Please upload tft.zip to /content/ before running."
    )

print("\nFiles in TFT directory:")
for f in sorted(os.listdir(extract_path)):
    f_size_mb = (extract_path / f).stat().st_size / (1024 * 1024)
    print(f"  - {f:<45} ({f_size_mb:.2f} MB)")


In [ ]:
# Cell 4: Load Sequence Archives via Memory Mapping
import numpy as np

tft_dir = "/content/tft"

# mmap_mode='r' streams batches directly without consuming gigabytes of system RAM
train_data = np.load(f"{tft_dir}/sequences_train.npz", mmap_mode="r")
val_data   = np.load(f"{tft_dir}/sequences_validation.npz", mmap_mode="r")
test_data  = np.load(f"{tft_dir}/sequences_test.npz", mmap_mode="r")

print("Sequence archives loaded successfully.")
print(f"Train sequences:      {len(train_data['y']):,}")
print(f"Validation sequences: {len(val_data['y']):,}")
print(f"Test sequences:       {len(test_data['y']):,}")


In [ ]:
# Cell 5: Inspect Input Shapes & Feature Lists
print("=== SHAPE VERIFICATION ===")
print(f"Train past_x:   {train_data['past_x'].shape} (expected [624451, 28, 18])")
print(f"Train future_x: {train_data['future_x'].shape} (expected [624451, 7, 14])")
print(f"Train y:        {train_data['y'].shape} (expected [624451, 7])")
print(f"Train y_raw:    {train_data['y_raw'].shape} (expected [624451, 7])")

past_feature_names = list(train_data["past_feature_names"])
future_feature_names = list(train_data["future_feature_names"])

print(f"\nPast Features ({len(past_feature_names)}):")
for i, name in enumerate(past_feature_names):
    print(f"  [{i:2d}] {name}")

print(f"\nFuture Features ({len(future_feature_names)}):")
for i, name in enumerate(future_feature_names):
    print(f"  [{i:2d}] {name}")


In [ ]:
# Cell 6: Build Categorical Entity Mappings (Training Set Only)
static_cols = [
    "store_nbr", "item_nbr", "family", "class",
    "perishable", "city", "state", "type", "cluster"
]

# Strict ML hygiene: build category vocabularies exclusively from training data
category_maps = {}
static_cardinalities = []

print("=== STATIC CATEGORY MAPPINGS ===")
for col in static_cols:
    unique_vals = np.unique(train_data[col])
    category_maps[col] = {val: idx for idx, val in enumerate(unique_vals)}
    cardinality = len(unique_vals)
    static_cardinalities.append(cardinality)
    print(f"  {col:<12}: {cardinality:3d} categories | Sample: {list(unique_vals[:5])}")

print(f"\nStatic Cardinalities List: {static_cardinalities}")


In [ ]:
# Cell 7: PyTorch Dataset & DataLoader Implementation
from torch.utils.data import Dataset, DataLoader

class TFTDataset(Dataset):
    """Memory-efficient PyTorch Dataset for Temporal Fusion Transformer."""
    def __init__(self, data, category_maps):
        self.past_x = data["past_x"]
        self.future_x = data["future_x"]
        self.y = data["y"]
        self.y_raw = data["y_raw"]
        self.static_data = {col: data[col] for col in static_cols}
        self.category_maps = category_maps
        self.length = len(self.y)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Continuous temporal features
        past_x_t   = torch.from_numpy(np.array(self.past_x[idx], dtype=np.float32))
        future_x_t = torch.from_numpy(np.array(self.future_x[idx], dtype=np.float32))
        y_t        = torch.from_numpy(np.array(self.y[idx], dtype=np.float32))
        y_raw_t    = torch.from_numpy(np.array(self.y_raw[idx], dtype=np.float32))

        # Static categorical features -> Integer IDs
        static_ids = [self.category_maps[col][self.static_data[col][idx]] for col in static_cols]
        static_x_t = torch.tensor(static_ids, dtype=torch.long)

        return {
            "past_x": past_x_t,
            "future_x": future_x_t,
            "static_x": static_x_t,
            "y": y_t,
            "y_raw": y_raw_t
        }

# Instantiate Datasets
train_dataset = TFTDataset(train_data, category_maps)
val_dataset   = TFTDataset(val_data, category_maps)
test_dataset  = TFTDataset(test_data, category_maps)

BATCH_SIZE = 64
NUM_WORKERS = 2  # Set to 2 for Colab multiprocessing

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"DataLoaders ready. Total Train Batches: {len(train_loader):,}, Val Batches: {len(val_loader)}")


In [ ]:
# Cell 8: Verify Single Batch Tensor Dimensions & Dtypes
sample_batch = next(iter(train_loader))

print("=== BATCH TENSOR VERIFICATION ===")
print(f"past_x:   {sample_batch['past_x'].shape:<25} dtype: {sample_batch['past_x'].dtype}")
print(f"future_x: {sample_batch['future_x'].shape:<25} dtype: {sample_batch['future_x'].dtype}")
print(f"static_x: {sample_batch['static_x'].shape:<25} dtype: {sample_batch['static_x'].dtype}")
print(f"y:        {sample_batch['y'].shape:<25} dtype: {sample_batch['y'].dtype}")
print(f"y_raw:    {sample_batch['y_raw'].shape:<25} dtype: {sample_batch['y_raw'].dtype}")

assert sample_batch['past_x'].shape == (BATCH_SIZE, 28, 18)
assert sample_batch['future_x'].shape == (BATCH_SIZE, 7, 14)
assert sample_batch['static_x'].shape == (BATCH_SIZE, 9)
assert sample_batch['y'].shape == (BATCH_SIZE, 7)
print("All tensor dimensions verified successfully.")


### Upgraded TFT Model Architecture
Key fixes from v2:
1. **Temporal Step Forecast Head**: Rather than discarding steps 0–5 and projecting only step 6, each future position in `final_output` is projected directly via `nn.Linear(hidden_size, 1)`.
2. **Dynamic Cardinalities**: Configurable embedding dimensions and hidden states.


In [ ]:
# Cell 9: Upgraded Temporal Fusion Transformer (TFT) Architecture
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# 1. Gated Residual Network (GRN)
# -----------------------------
class GatedResidualNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1):
        super().__init__()
        output_size = output_size or hidden_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.gate = nn.Linear(output_size, output_size)
        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(output_size)

    def forward(self, x):
        residual = self.skip(x)
        h = F.elu(self.fc1(x))
        h = self.dropout(h)
        h = self.fc2(h)
        gate = torch.sigmoid(self.gate(h))
        return self.norm(gate * h + (1 - gate) * residual)

# -----------------------------
# 2. Variable Selection Network (VSN) for Temporal Features
# -----------------------------
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_variables, input_size, hidden_size, dropout=0.1):
        super().__init__()
        self.num_variables = num_variables
        self.variable_grns = nn.ModuleList([
            GatedResidualNetwork(input_size, hidden_size, hidden_size, dropout=dropout)
            for _ in range(num_variables)
        ])
        self.selection_grn = GatedResidualNetwork(
            num_variables * input_size, hidden_size, num_variables, dropout=dropout
        )

    def forward(self, x):
        # x: [B, T, num_variables, input_size]
        B, T, V, E = x.shape
        var_outputs = [self.variable_grns[i](x[:, :, i, :]) for i in range(V)]
        var_outputs = torch.stack(var_outputs, dim=2)  # [B, T, V, hidden_size]

        flattened = x.reshape(B, T, V * E)
        weights = torch.softmax(self.selection_grn(flattened), dim=-1).unsqueeze(-1)
        selected = torch.sum(weights * var_outputs, dim=2)  # [B, T, hidden_size]
        return selected, weights.squeeze(-1)

# -----------------------------
# 3. Static Variable Selection Network
# -----------------------------
class StaticVariableSelectionNetwork(nn.Module):
    def __init__(self, num_variables, embedding_size, hidden_size, dropout=0.1):
        super().__init__()
        self.num_variables = num_variables
        self.variable_grns = nn.ModuleList([
            GatedResidualNetwork(embedding_size, hidden_size, hidden_size, dropout=dropout)
            for _ in range(num_variables)
        ])
        self.selection_grn = GatedResidualNetwork(
            num_variables * embedding_size, hidden_size, num_variables, dropout=dropout
        )

    def forward(self, x):
        # x: [B, num_variables, embedding_size]
        B, V, E = x.shape
        outputs = [self.variable_grns[i](x[:, i, :]) for i in range(V)]
        outputs = torch.stack(outputs, dim=1)  # [B, V, hidden_size]

        flattened = x.reshape(B, V * E)
        weights = torch.softmax(self.selection_grn(flattened), dim=-1)
        selected = torch.sum(weights.unsqueeze(-1) * outputs, dim=1)  # [B, hidden_size]
        return selected, weights

# -----------------------------
# 4. Full TFT Model Class
# -----------------------------
class TFT(nn.Module):
    def __init__(
        self,
        past_features=18,
        future_features=14,
        static_cardinalities=None,
        hidden_size=64,
        embedding_size=16,
        num_heads=4,
        dropout=0.1,
        forecast_horizon=7
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.forecast_horizon = forecast_horizon

        # Static Entity Embeddings
        self.static_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_size)
            for cardinality in static_cardinalities
        ])
        self.static_vsn = StaticVariableSelectionNetwork(
            num_variables=len(static_cardinalities),
            embedding_size=embedding_size,
            hidden_size=hidden_size,
            dropout=dropout
        )

        # Temporal Projections
        self.past_variable_embeddings = nn.ModuleList([
            nn.Linear(1, hidden_size) for _ in range(past_features)
        ])
        self.future_variable_embeddings = nn.ModuleList([
            nn.Linear(1, hidden_size) for _ in range(future_features)
        ])

        # Temporal Variable Selection Networks
        self.past_vsn = VariableSelectionNetwork(past_features, hidden_size, hidden_size, dropout=dropout)
        self.future_vsn = VariableSelectionNetwork(future_features, hidden_size, hidden_size, dropout=dropout)

        # LSTM Encoder & Decoder
        self.encoder_lstm = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)

        # Static Enrichment
        self.static_enrichment = GatedResidualNetwork(hidden_size * 2, hidden_size, hidden_size, dropout=dropout)

        # Multi-Head Temporal Attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.attention_norm = nn.LayerNorm(hidden_size)

        # Final Processing & Forecast Head
        self.final_grn = GatedResidualNetwork(hidden_size, hidden_size, hidden_size, dropout=dropout)
        
        # UPGRADED FORECAST HEAD: Linear(hidden_size, 1) per temporal step
        self.forecast_head = nn.Linear(hidden_size, 1)

    def embed_static(self, static_x):
        embeddings = [embed(static_x[:, i]) for i, embed in enumerate(self.static_embeddings)]
        return torch.stack(embeddings, dim=1)

    def embed_temporal(self, x, embeddings):
        outputs = [embed(x[:, :, i:i+1]) for i, embed in enumerate(embeddings)]
        return torch.stack(outputs, dim=2)

    def forward(self, past_x, future_x, static_x):
        # 1. Static Selection
        static_embedded = self.embed_static(static_x)
        static_context, _ = self.static_vsn(static_embedded)

        # 2. Past Variable Selection & Encoding
        past_embedded = self.embed_temporal(past_x, self.past_variable_embeddings)
        past_selected, _ = self.past_vsn(past_embedded)
        _, (h, c) = self.encoder_lstm(past_selected)

        # 3. Future Variable Selection & Decoding
        future_embedded = self.embed_temporal(future_x, self.future_variable_embeddings)
        future_selected, _ = self.future_vsn(future_embedded)
        decoder_output, _ = self.decoder_lstm(future_selected, (h, c))

        # 4. Static Enrichment
        static_repeated = static_context.unsqueeze(1).expand(-1, decoder_output.size(1), -1)
        enriched = self.static_enrichment(torch.cat([decoder_output, static_repeated], dim=-1))

        # 5. Multi-Head Temporal Attention
        attn_out, _ = self.attention(enriched, enriched, enriched)
        attn_norm = self.attention_norm(attn_out + enriched)

        # 6. Final GRN & Multi-Step Output
        final_output = self.final_grn(attn_norm)  # [B, 7, hidden_size]
        
        # Squeeze last dimension: [B, 7, 1] -> [B, 7]
        output = self.forecast_head(final_output).squeeze(-1)
        return output


In [ ]:
# Cell 10: Verify Model Instantiation and Parameter Count
test_model = TFT(
    past_features=18,
    future_features=14,
    static_cardinalities=static_cardinalities,
    hidden_size=64,
    embedding_size=16,
    num_heads=4,
    dropout=0.1,
    forecast_horizon=7
).to(device)

test_model.eval()
with torch.no_grad():
    test_out = test_model(
        sample_batch["past_x"].to(device),
        sample_batch["future_x"].to(device),
        sample_batch["static_x"].to(device)
    )

param_count = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Test forward pass output shape: {test_out.shape} (dtype: {test_out.dtype})")
print(f"Total trainable parameters:     {param_count:,}")
assert test_out.shape == (BATCH_SIZE, 7)
del test_model, test_out
torch.cuda.empty_cache()


In [ ]:
# Cell 11: Load Target Sales Scaler for Inverting Predictions
import joblib

scaler_path = "/content/target_sales_scaler.joblib"
if not os.path.exists(scaler_path):
    scaler_path = f"{tft_dir}/target_sales_scaler.joblib"

scaler_dict = joblib.load(scaler_path)
target_scaler = scaler_dict["target_scaler"]

print("Target scaler loaded successfully:")
print(f"  Mean:  {target_scaler.mean_[0]:.7f}")
print(f"  Scale: {target_scaler.scale_[0]:.7f}")


### Optuna Hyperparameter Optimization
- **Loss Function**: `nn.HuberLoss(delta=...)` (aligns gradients with linear demand metrics like WAPE and MAE).
- **Early Pruning**: Uses `MedianPruner` to terminate unpromising trials after epoch 1 or 2.
- **Epochs per Trial**: 3 epochs (fast exploration).


In [ ]:
# Cell 12: Optuna Objective Function Definition
import torch.optim as optim

def objective(trial):
    # -----------------------------
    # 1. Hyperparameter Search Space
    # -----------------------------
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 96])
    
    # num_heads must divide hidden_size
    valid_heads = [h for h in [2, 4, 8] if hidden_size % h == 0]
    num_heads = trial.suggest_categorical("num_heads", valid_heads)
    
    embedding_size = trial.suggest_categorical("embedding_size", [8, 16, 24])
    dropout = trial.suggest_float("dropout", 0.05, 0.25, step=0.05)
    lr = trial.suggest_float("lr", 1e-4, 6e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 5e-4, log=True)
    huber_delta = trial.suggest_categorical("huber_delta", [0.75, 1.0, 1.25])

    # -----------------------------
    # 2. Instantiate Model & Optimizer
    # -----------------------------
    model = TFT(
        past_features=18,
        future_features=14,
        static_cardinalities=static_cardinalities,
        hidden_size=hidden_size,
        embedding_size=embedding_size,
        num_heads=num_heads,
        dropout=dropout,
        forecast_horizon=7
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.HuberLoss(delta=huber_delta)

    # Fast search: 3 epochs per trial is sufficient to rank configurations
    SEARCH_EPOCHS = 3

    for epoch in range(1, SEARCH_EPOCHS + 1):
        # Training loop
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            past_x   = batch["past_x"].to(device, non_blocking=True)
            future_x = batch["future_x"].to(device, non_blocking=True)
            static_x = batch["static_x"].to(device, non_blocking=True)
            y        = batch["y"].to(device, non_blocking=True)

            optimizer.zero_grad()
            preds = model(past_x, future_x, static_x)
            loss = criterion(preds, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation loop
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                past_x   = batch["past_x"].to(device, non_blocking=True)
                future_x = batch["future_x"].to(device, non_blocking=True)
                static_x = batch["static_x"].to(device, non_blocking=True)
                y        = batch["y"].to(device, non_blocking=True)

                preds = model(past_x, future_x, static_x)
                val_loss += criterion(preds, y).item()

        val_loss /= len(val_loader)

        # Report to Optuna for early pruning
        trial.report(val_loss, epoch)
        if trial.should_prune():
            del model, optimizer
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

    del model, optimizer
    torch.cuda.empty_cache()
    return val_loss


In [ ]:
# Cell 13: Run Optuna Study (10 Trials with Median Pruning)
from optuna.pruners import MedianPruner

pruner = MedianPruner(n_startup_trials=3, n_warmup_steps=1)
study = optuna.create_study(direction="minimize", pruner=pruner, study_name="tft_favorita_optuna")

print("Starting Optuna Study (10 trials)...")
study.optimize(objective, n_trials=10, timeout=None)

print("\n" + "="*50)
print("OPTUNA STUDY COMPLETE")
print("="*50)
print(f"Best Trial Number: {study.best_trial.number}")
print(f"Best Validation Huber Loss: {study.best_value:.6f}")
print("\nBest Hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k:<15}: {v}")


### Train Best TFT Model to Full Convergence
Trains the winning hyperparameter configuration with early stopping (patience = 5).


In [ ]:
# Cell 14: Train Winning TFT Model
import time

best_p = study.best_params

final_model = TFT(
    past_features=18,
    future_features=14,
    static_cardinalities=static_cardinalities,
    hidden_size=best_p["hidden_size"],
    embedding_size=best_p["embedding_size"],
    num_heads=best_p["num_heads"],
    dropout=best_p["dropout"],
    forecast_horizon=7
).to(device)

final_optimizer = optim.AdamW(
    final_model.parameters(),
    lr=best_p["lr"],
    weight_decay=best_p["weight_decay"]
)
final_criterion = nn.HuberLoss(delta=best_p["huber_delta"])

MAX_EPOCHS = 25
PATIENCE = 5
patience_counter = 0
best_val_loss = float("inf")
checkpoint_path = "/content/tft_optuna_best_model.pth"

print(f"Training final model with best parameters for up to {MAX_EPOCHS} epochs...")

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    
    # Train
    final_model.train()
    train_loss = 0.0
    for batch in train_loader:
        past_x   = batch["past_x"].to(device, non_blocking=True)
        future_x = batch["future_x"].to(device, non_blocking=True)
        static_x = batch["static_x"].to(device, non_blocking=True)
        y        = batch["y"].to(device, non_blocking=True)

        final_optimizer.zero_grad()
        preds = final_model(past_x, future_x, static_x)
        loss = final_criterion(preds, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        final_optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    final_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            past_x   = batch["past_x"].to(device, non_blocking=True)
            future_x = batch["future_x"].to(device, non_blocking=True)
            static_x = batch["static_x"].to(device, non_blocking=True)
            y        = batch["y"].to(device, non_blocking=True)

            preds = final_model(past_x, future_x, static_x)
            val_loss += final_criterion(preds, y).item()
    val_loss /= len(val_loader)

    elapsed = time.time() - t0

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(final_model.state_dict(), checkpoint_path)
        status = "✓ BEST CHECKPOINT SAVED"
    else:
        patience_counter += 1
        status = f"no improvement ({patience_counter}/{PATIENCE})"

    print(f"Epoch {epoch:02d} | Train Huber: {train_loss:.6f} | Val Huber: {val_loss:.6f} | Time: {elapsed:.1f}s | {status}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

print(f"\nTraining complete! Best Validation Huber Loss: {best_val_loss:.6f}")


In [ ]:
# Cell 15: Evaluate Best Model on Unseen Test Split (2017-08-01 to 2017-08-15)
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load best checkpoint
final_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
final_model.eval()

all_preds = []
all_true  = []

with torch.no_grad():
    for batch in test_loader:
        past_x   = batch["past_x"].to(device)
        future_x = batch["future_x"].to(device)
        static_x = batch["static_x"].to(device)

        # 1. Predict in standardized log-space
        pred_scaled = final_model(past_x, future_x, static_x).cpu().numpy()

        # 2. Invert standard scaler: scaled -> log1p(sales)
        pred_log = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1))

        # 3. Invert log1p: log1p -> actual sales demand
        pred_sales = np.expm1(pred_log).reshape(pred_scaled.shape)
        
        # 4. Non-negative demand constraint
        pred_sales = np.maximum(pred_sales, 0.0)

        true_sales = batch["y_raw"].cpu().numpy()

        all_preds.append(pred_sales)
        all_true.append(true_sales)

preds = np.concatenate(all_preds, axis=0)
true  = np.concatenate(all_true, axis=0)

# Calculate Overall Test Metrics
mae = mean_absolute_error(true.flatten(), preds.flatten())
rmse = np.sqrt(mean_squared_error(true.flatten(), preds.flatten()))
wape = (np.sum(np.abs(true - preds)) / np.sum(np.abs(true))) * 100.0
smape = np.mean(2.0 * np.abs(preds - true) / (np.abs(true) + np.abs(preds) + 1e-8)) * 100.0

print("=" * 55)
print("FINAL OPTUNA-OPTIMIZED TFT TEST RESULTS")
print("=" * 55)
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"WAPE  : {wape:.2f}%")
print(f"sMAPE : {smape:.2f}%")
print("=" * 55)

print("\nDAY-BY-DAY HORIZON BREAKDOWN (Day 1 -> Day 7):")
for d in range(7):
    day_t = true[:, d]
    day_p = preds[:, d]
    d_mae  = mean_absolute_error(day_t, day_p)
    d_rmse = np.sqrt(mean_squared_error(day_t, day_p))
    d_wape = (np.sum(np.abs(day_t - day_p)) / np.sum(np.abs(day_t))) * 100.0
    print(f"  Day {d+1}: MAE = {d_mae:7.4f} | RMSE = {d_rmse:7.4f} | WAPE = {d_wape:5.2f}%")


In [ ]:
# Cell 16: Download Best Model Checkpoint
try:
    from google.colab import files
    files.download(checkpoint_path)
    print("Download initiated.")
except Exception as e:
    print(f"Saved locally at {checkpoint_path}.")
